#  Ajustement de Cercle — Méthode des Moindres Carrés Linéaires

> **Objectif :** Étant donné un nuage de points (potentiellement bruités), retrouver le centre $(x_0, y_0)$ et le rayon $r$ du cercle qui les ajuste au mieux.

---
## 1.  Principe Mathématique — Linéarisation

L'équation générale d'un cercle de centre $(x_0, y_0)$ et de rayon $r$ est :

$$
(x - x_0)^2 + (y - y_0)^2 = r^2
$$

**Développement :**

$$
x^2 - 2x x_0 + x_0^2 + y^2 - 2y y_0 + y_0^2 = r^2
$$

On regroupe les termes connus $(x^2 + y^2)$ à droite :

$$
2x \cdot x_0 + 2y \cdot y_0 + (r^2 - x_0^2 - y_0^2) = x^2 + y^2
$$

###  Substitution — Passage au système linéaire

On introduit trois nouvelles inconnues pour **linéariser** le problème :

| Inconnue | Définition | Récupération finale |
|:--------:|:----------:|:-------------------:|
| $a$ | $-2x_0$ | $x_0 = -a/2$ |
| $b$ | $-2y_0$ | $y_0 = -b/2$ |
| $c$ | $x_0^2 + y_0^2 - r^2$ | $r = \sqrt{x_0^2 + y_0^2 - c}$ |

Le système devient **linéaire** en $(a, b, c)$ :

$$
\boxed{a \cdot x + b \cdot y + c = -(x^2 + y^2)}
$$

---
## 2.  Formulation Matricielle

Pour $n$ points $(x_i, y_i)$, on écrit le système $A\,\theta = \mathbf{b}$ :

$$
\underbrace{\begin{bmatrix} x_1 & y_1 & 1 \\ x_2 & y_2 & 1 \\ \vdots & \vdots & \vdots \\ x_n & y_n & 1 \end{bmatrix}}_{A \in \mathbb{R}^{n \times 3}}
\underbrace{\begin{bmatrix} a \\ b \\ c \end{bmatrix}}_{\theta}
=
\underbrace{\begin{bmatrix} -(x_1^2 + y_1^2) \\ -(x_2^2 + y_2^2) \\ \vdots \\ -(x_n^2 + y_n^2) \end{bmatrix}}_{\mathbf{b}}
$$

Comme le système est **sur-déterminé** ($n \gg 3$), on le résout au sens des **moindres carrés** :

$$
\theta^* = \arg\min_\theta \|A\theta - \mathbf{b}\|^2
$$

Ce qui correspond à la solution de l'équation normale : $A^T A\, \theta = A^T \mathbf{b}$.

---
## 3.  Implémentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def methode_lineaire(x, y):
    """
    Ajuste un cercle à un nuage de points par moindres carrés linéaires.

    Principe : on linéarise (x-x0)² + (y-y0)² = r²
    sous la forme : a·x + b·y + c = -(x² + y²)
    avec a = -2x0,  b = -2y0,  c = x0²+y0²-r²

    Paramètres
    ----------
    x, y : array — coordonnées des points

    Retourne
    --------
    x0, y0 : float — centre estimé
    r      : float — rayon estimé
    """

    # Matrice A : chaque ligne = [xi, yi, 1] pour le point i
    A = np.column_stack((x, y, np.ones(len(x))))

    # Vecteur b : membre droit après passage de x²+y² à gauche
    b = -(x**2 + y**2)

    # Résolution par moindres carrés
    a_coef, b_coef, c_coef = np.linalg.lstsq(A, b, rcond=None)[0]

    # Récupération du centre
    # On avait posé a = -2·x0  →  x0 = -a/2
    # On avait posé b = -2·y0  →  y0 = -b/2
    x0 = -a_coef / 2
    y0 = -b_coef / 2

    # Récupération du rayon
    # On avait posé c = x0²+y0²-r²  →  r² = x0²+y0²-c
    r = np.sqrt(x0**2 + y0**2 - c_coef)

    return x0, y0, r

---
## 4.  Test sur un cercle parfait (sans bruit)

Cercle de référence : centre $(2,\,-1)$, rayon $r = 5$.

In [ ]:
theta = np.linspace(0, 2*np.pi, 50)
x_exact = 2 + 5 * np.cos(theta)
y_exact = -1 + 5 * np.sin(theta)

x0, y0, r = methode_lineaire(x_exact, y_exact)

print(f"Centre estimé : ({x0:.6f}, {y0:.6f})   — attendu : (2.0, -1.0)")
print(f"Rayon estimé  :  {r:.6f}              — attendu :  5.0")

---
## 5.  Test sur des points bruités

On génère $n = 100$ points sur un cercle de rayon $r = 3$ centré à l'origine, puis on ajoute un bruit gaussien :

$$
\varepsilon \sim \mathcal{N}\bigl(0,\, \sigma\bigr), \quad \sigma = 0.5
$$

In [ ]:
# Paramètres
r_vrai = 3
n = 100
sigma = 0.5

np.random.seed(42)  # reproductibilité

# Points sur le cercle (angles aléatoires)
theta = np.random.uniform(0, 2*np.pi, n)
x = r_vrai * np.cos(theta)
y = r_vrai * np.sin(theta)

# Bruit gaussien sur x et y
bruit = np.random.normal(loc=0, scale=np.sqrt(sigma), size=(n, 2))
x_bruite = x + bruit[:, 0]
y_bruite = y + bruit[:, 1]

# Ajustement
xc, yc, r_estime = methode_lineaire(x_bruite, y_bruite)

print(f"Centre estimé : ({xc:.4f}, {yc:.4f})   — attendu : (0.0, 0.0)")
print(f"Rayon estimé  :  {r_estime:.4f}          — attendu :  {r_vrai}")

---
## 6.  Visualisation

In [ ]:
theta_plot = np.linspace(0, 2*np.pi, 300)

# Cercle estimé
x_cercle = xc + r_estime * np.cos(theta_plot)
y_cercle = yc + r_estime * np.sin(theta_plot)

# Cercle vrai
x_vrai_plot = r_vrai * np.cos(theta_plot)
y_vrai_plot = r_vrai * np.sin(theta_plot)

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(x_bruite, y_bruite, alpha=0.5, color='steelblue', label='Points bruités')
ax.plot(x_vrai_plot, y_vrai_plot, 'g--', linewidth=2, label=f'Cercle vrai  (r = {r_vrai})')
ax.plot(x_cercle, y_cercle, 'r-', linewidth=2, label=f'Cercle estimé (r = {r_estime:.2f})')
ax.plot(xc, yc, 'r+', markersize=14, markeredgewidth=2.5,
        label=f'Centre estimé ({xc:.2f}, {yc:.2f})')

ax.set_aspect('equal')
ax.set_title('Ajustement de cercle — Méthode linéaire', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

---
## 7.  Résumé

| Étape | Action |
|:-----:|:-------|
| **1** | Développer $(x-x_0)^2 + (y-y_0)^2 = r^2$ |
| **2** | Poser $a=-2x_0$, $b=-2y_0$, $c=x_0^2+y_0^2-r^2$ pour **linéariser** |
| **3** | Construire $A = [x,\; y,\; 1]$ et $\mathbf{b} = -(x^2+y^2)$ |
| **4** | Résoudre $A\theta = \mathbf{b}$ par **moindres carrés** (`np.linalg.lstsq`) |
| **5** | Récupérer $x_0 = -a/2$, $\;y_0 = -b/2$, $\;r = \sqrt{x_0^2+y_0^2-c}$ |

---

> **Avantage :** Solution directe, pas d'optimisation itérative, très rapide.  
> 